# Shared TX + Multilingual Receiver — using DeepSC `all` functions

This version intentionally reuses the functions already present in the **DeepSC `all` branch** instead of duplicating them inside the notebook.

Reused directly from the repository:

- `EurMultilingualDatasetBPE`
- `collate_multilingual_bpe`
- `load_bpe`
- `SeqtoTextBPE`
- `transmitter`
- `greedy_decode`
- `beam_decode`
- `send_through_channel`
- `load_teacher`
- `setup_seed`
- `DeepSCMetrics`

The only experiment-specific logic kept here is:

```text
src
 ↓
transmitter(...)
 ↓
Tx_sig
 ↓
send_through_channel(...)   ← ONCE
 ↓
Rx_sig
 ├── greedy/beam <EN>
 ├── greedy/beam <PT>
 ├── greedy/beam <ES>
 └── greedy/beam <FR>
```

The **same `Rx_sig`** is reused for all target languages.


## 1. Imports

Run this notebook from the root of the DeepSC repository while checked out on branch `all`.


In [1]:
from __future__ import annotations

import json
import os
from functools import partial
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dataset_multilingual import (
    EurMultilingualDatasetBPE,
    collate_multilingual_bpe,
)

from models.transceiver import DeepSC

from utils.bpe_utils import (
    SeqtoTextBPE,
    beam_decode,
    greedy_decode,
    load_bpe,
    transmitter,
)

from utils.model_utils import (
    load_teacher,
    send_through_channel,
    setup_seed,
)

from metrics.metrics_bpe import DeepSCMetrics


/home/prinako/External/WorkSpace/PGCC/DeepSC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

In [11]:
# ------------------------------------------------------------
# Dataset produced by process_multilingual_deepsc.py
# Use the EXACT preprocessing directory.
# ------------------------------------------------------------
DATA_DIR = "data/train/europarl_bpe/2026-09-19"

# ------------------------------------------------------------
# Teacher/model checkpoints
# ------------------------------------------------------------
ENCODER_CHECKPOINT = "results/base_model/2026-09-19/encoder_17.pth"
DECODER_CHECKPOINT = "results/base_model/2026-09-19/decoder_17.pth"

# Optional teacher run config from main_multi_vocab.py
RUN_CONFIG = None
# Example:
# RUN_CONFIG = "checkpoints/.../run_config.json"

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
SPLIT = "val"                   # use "test" only for final evaluation
LANGUAGES = ("en", "pt", "es", "fr")

CHANNEL = "Rayleigh"
SNR_DB = [0, 4, 8, 12, 16]
TRIALS = 1

DECODE = "greedy"               # "greedy" or "beam"
BEAM_SIZE = 5
LENGTH_PENALTY = 0.7

BATCH_SIZE = 32
NUM_WORKERS = 0
SEED = 48

MAX_SAMPLES = 20                # None = whole split

# Metrics can be expensive because semantic similarity loads a transformer.
COMPUTE_METRICS = True
SEMANTIC_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

OUTPUT_DIR = "results/multilingual_shared_tx"


## 3. Read preprocessing/model configuration

The new synchronized preprocessing stores the authoritative sequence length in `preprocessing_config.json`.


In [3]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


preprocessing_path = os.path.join(
    DATA_DIR,
    "preprocessing_config.json",
)

vocab_path = os.path.join(
    DATA_DIR,
    "vocab_bpe.json",
)

tokenizer_path = os.path.join(
    DATA_DIR,
    "tokenizer_bpe.model",
)

preprocessing_config = read_json(preprocessing_path)

if RUN_CONFIG is None:
    candidate = os.path.join(
        os.path.dirname(ENCODER_CHECKPOINT),
        "run_config.json",
    )
    run_config = read_json(candidate) if os.path.exists(candidate) else {}
else:
    run_config = read_json(RUN_CONFIG)

MAX_LEN = int(
    run_config.get(
        "max_length",
        preprocessing_config.get("max_seq_len", 64),
    )
)

NUM_LAYERS = int(run_config.get("num_layers", 8))
NUM_HEADS = int(run_config.get("num_heads", 16))
D_MODEL = int(run_config.get("d_model", 128))
DFF = int(run_config.get("dff", 512))
DROPOUT = float(run_config.get("dropout", 0.1))

print("Preprocessing max_seq_len:", preprocessing_config.get("max_seq_len"))
print("Model max length         :", MAX_LEN)
print("Teacher layers / heads   :", NUM_LAYERS, "/", NUM_HEADS)


Preprocessing max_seq_len: 64
Model max length         : 64
Teacher layers / heads   : 8 / 16


## 4. Load BPE with the repository helper

`load_bpe()` already checks the fixed IDs for PAD/START/END/UNK and EN/PT/ES/FR.


In [4]:
token_to_idx, sp = load_bpe(
    vocab_path,
    tokenizer_path,
)

VOCAB_SIZE = len(token_to_idx)

PAD_IDX = int(token_to_idx["<PAD>"])
START_IDX = int(token_to_idx["<START>"])
END_IDX = int(token_to_idx["<END>"])

LANG_ID = {
    "en": int(token_to_idx["<EN>"]),
    "pt": int(token_to_idx["<PT>"]),
    "es": int(token_to_idx["<ES>"]),
    "fr": int(token_to_idx["<FR>"]),
}

converter = SeqtoTextBPE(
    vocb_dictionary=token_to_idx,
    end_idx=END_IDX,
    tokenizer_model=tokenizer_path,
)

print("Vocabulary size:", VOCAB_SIZE)
print("Language IDs:", LANG_ID)


BPE CHECK
SentencePiece vocab : 32000
JSON vocab          : 32000
<PAD>    JSON=0     SP=0    
<START>  JSON=1     SP=1    
<END>    JSON=2     SP=2    
<UNK>    JSON=3     SP=3    
<EN>     JSON=4     SP=4    
<PT>     JSON=5     SP=5    
<ES>     JSON=6     SP=6    
<FR>     JSON=7     SP=7    
Vocabulary size: 32000
Language IDs: {'en': 4, 'pt': 5, 'es': 6, 'fr': 7}


## 5. Load the synchronized dataset

This directly reuses `EurMultilingualDatasetBPE` and `collate_multilingual_bpe`.


In [5]:
dataset = EurMultilingualDatasetBPE(
    split=SPLIT,
    data_dir=DATA_DIR,
    languages=LANGUAGES,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=partial(
        collate_multilingual_bpe,
        pad_idx=PAD_IDX,
    ),
)

print("Split:", SPLIT)
print("Samples:", len(dataset))


Split: val
Samples: 231881


## 6. Build and load the Teacher

The repository's `load_teacher()` loads:

- semantic encoder;
- channel encoder;
- channel decoder;
- semantic decoder;
- output projection.


In [6]:
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

setup_seed(SEED)

model = DeepSC(
    NUM_LAYERS,
    VOCAB_SIZE,
    VOCAB_SIZE,
    MAX_LEN,
    MAX_LEN,
    D_MODEL,
    NUM_HEADS,
    DFF,
    DROPOUT,
).to(device)

model = load_teacher(
    model,
    ENCODER_CHECKPOINT,
    DECODER_CHECKPOINT,
    device,
)

model.eval()

print("Device:", device)


Teacher loaded
Device: cuda:0


## 7. Inspect one aligned sample


In [7]:
sample = dataset[0]

print("ID:", sample["id"])
print("Shared TX EN:")
print(converter.sequence_to_text(sample["src"]))

print()

for lang in LANGUAGES:
    print(
        f"{lang.upper()} reference:",
        converter.sequence_to_text(
            sample["targets"][lang]
        ),
    )


ID: 312325
Shared TX EN:
to do so would be like saying to someone bent upon murder, 'please don't commit murder, but if you do, there will be no criminal proceedings anyway'.

EN reference: to do so would be like saying to someone bent upon murder, 'please don't commit murder, but if you do, there will be no criminal proceedings anyway'.
PT reference: fazer isso seria como dizer a alguém decidido a assassinar: 'por favor, não cometa assassinato, mas se o fizer, não haverá qualquer processo criminal'.
ES reference: hacerlo sería como decirle a alguien decidido a asesinar, 'por favor, no cometa asesinato, pero si lo hace, no habrá ningún procedimiento penal de todos modos'.
FR reference: ce serait comme dire à quelqu'un déterminé à tuer: "s'il vous plaît, ne commettez pas de meurtre, mais si vous le faites, il n'y aura pas de procédure pénale de toute façon".


## 8. Core experiment: transmit once, decode four times

This cell uses the repository's existing `transmitter()` and `send_through_channel()` functions.

The important part is:

```python
tx_sig = transmitter(...)
rx_sig = send_through_channel(...)   # called once

for lang in LANGUAGES:
    decode(rx_sig, target_language_idx=...)
```


In [8]:
batch = next(iter(loader))

src = batch["src"][:1].to(device)

src_mask = (
    (src == PAD_IDX)
    .unsqueeze(-2)
    .to(device=device, dtype=torch.float32)
)

snr_db = float(SNR_DB[0])

# ============================================================
# SHARED TX — repository function
# ============================================================
tx_sig = transmitter(
    src=src,
    src_mask=src_mask,
    model=model,
    padding_idx=PAD_IDX,
    device=device,
)

# ============================================================
# CHANNEL — exactly ONE realization
# ============================================================
rx_sig = send_through_channel(
    tx_sig,
    CHANNEL,
    snr_db,
)

print("Source:")
print(converter.sequence_to_text(src[0].cpu()))
print()
print("Tx_sig shape:", tuple(tx_sig.shape))
print("Rx_sig shape:", tuple(rx_sig.shape))

rows = []

for lang in LANGUAGES:

    common_args = dict(
        Rx_sig=rx_sig,             # SAME Rx_sig
        model=model,
        src=src,
        src_mask=src_mask,
        max_len=MAX_LEN,
        start_idx=START_IDX,
        padding_idx=PAD_IDX,
        end_idx=END_IDX,
        target_language_idx=LANG_ID[lang],
        channel=CHANNEL,
        snr=snr_db,
        device=device,
    )

    if DECODE == "greedy":
        output = greedy_decode(**common_args)

    elif DECODE == "beam":
        forbidden = {
            PAD_IDX,
            START_IDX,
            LANG_ID["en"],
            LANG_ID["pt"],
            LANG_ID["es"],
            LANG_ID["fr"],
        }

        output = beam_decode(
            **common_args,
            beam_size=BEAM_SIZE,
            length_penalty=LENGTH_PENALTY,
            forbidden_token_ids=forbidden,
            return_all_beams=False,
        )

    else:
        raise ValueError("DECODE must be 'greedy' or 'beam'.")

    prediction = converter.sequence_to_text(
        output[0].detach().cpu()
    )

    reference = converter.sequence_to_text(
        batch["targets"][lang][0]
    )

    rows.append(
        {
            "language": lang.upper(),
            "reference": reference,
            "prediction": prediction,
        }
    )

pd.DataFrame(rows)


Source:
to do so would be like saying to someone bent upon murder, 'please don't commit murder, but if you do, there will be no criminal proceedings anyway'.

Tx_sig shape: (1, 38, 16)
Rx_sig shape: (1, 38, 16)


,language,reference,prediction
0,EN,to do so would be like saying to someone bent ...,"it to do to be as which to go part-in to pass,..."
1,PT,fazer isso seria como dizer a alguém decidido ...,"por isso, gostaria de dizer que, em vez, a par..."
2,ES,hacerlo sería como decirle a alguien decidido ...,"por lo tanto, me gustaría decir que se refiere..."
3,FR,ce serait comme dire à quelqu'un déterminé à t...,"il s'agit d'un fait que, en tant que bin-à-dir..."


## 9. Dataset evaluation loop

The channel is sampled once per batch/SNR/trial, then the same received tensor is reused across all languages.


In [9]:
prediction_rows = []

for snr_db in SNR_DB:

    for trial in range(TRIALS):

        trial_seed = (
            SEED
            + int(float(snr_db) * 1000)
            + trial * 1_000_003
        )

        setup_seed(trial_seed)

        processed = 0

        pbar = tqdm(
            loader,
            desc=f"SNR={snr_db:g} dB | trial={trial + 1}",
        )

        for batch in pbar:

            if MAX_SAMPLES is not None:
                remaining = MAX_SAMPLES - processed

                if remaining <= 0:
                    break
            else:
                remaining = None

            ids = batch["ids"]
            src = batch["src"]
            targets = batch["targets"]

            if (
                remaining is not None
                and src.size(0) > remaining
            ):
                ids = ids[:remaining]
                src = src[:remaining]

                targets = {
                    lang: tensor[:remaining]
                    for lang, tensor in targets.items()
                }

            src = src.to(
                device,
                non_blocking=True,
            )

            src_mask = (
                (src == PAD_IDX)
                .unsqueeze(-2)
                .to(
                    device=device,
                    dtype=torch.float32,
                )
            )

            # ========================================================
            # 1. SHARED TRANSMITTER
            # ========================================================
            tx_sig = transmitter(
                src=src,
                src_mask=src_mask,
                model=model,
                padding_idx=PAD_IDX,
                device=device,
            )

            # ========================================================
            # 2. ONE CHANNEL REALIZATION
            # ========================================================
            rx_sig = send_through_channel(
                tx_sig,
                CHANNEL,
                float(snr_db),
            )

            source_texts = [
                converter.sequence_to_text(row)
                for row in src.detach().cpu()
            ]

            # ========================================================
            # 3. SAME Rx_sig → EN/PT/ES/FR
            # ========================================================
            for lang in LANGUAGES:

                common_args = dict(
                    Rx_sig=rx_sig,
                    model=model,
                    src=src,
                    src_mask=src_mask,
                    max_len=MAX_LEN,
                    start_idx=START_IDX,
                    padding_idx=PAD_IDX,
                    end_idx=END_IDX,
                    target_language_idx=LANG_ID[lang],
                    channel=CHANNEL,
                    snr=float(snr_db),
                    device=device,
                )

                if DECODE == "greedy":

                    generated = greedy_decode(
                        **common_args
                    )

                else:

                    forbidden = {
                        PAD_IDX,
                        START_IDX,
                        *LANG_ID.values(),
                    }

                    generated = beam_decode(
                        **common_args,
                        beam_size=BEAM_SIZE,
                        length_penalty=LENGTH_PENALTY,
                        forbidden_token_ids=forbidden,
                        return_all_beams=False,
                    )

                predictions = [
                    converter.sequence_to_text(row)
                    for row in generated.detach().cpu()
                ]

                references = [
                    converter.sequence_to_text(row)
                    for row in targets[lang]
                ]

                for (
                    sample_id,
                    source_text,
                    reference,
                    prediction,
                ) in zip(
                    ids,
                    source_texts,
                    references,
                    predictions,
                ):

                    prediction_rows.append(
                        {
                            "id": sample_id,
                            "snr_db": float(snr_db),
                            "trial": trial + 1,
                            "language": lang,
                            "source_en": source_text,
                            "reference": reference,
                            "prediction": prediction,
                        }
                    )

            processed += src.size(0)

            pbar.set_postfix(
                samples=processed
            )

            if (
                MAX_SAMPLES is not None
                and processed >= MAX_SAMPLES
            ):
                break


predictions_df = pd.DataFrame(
    prediction_rows
)

predictions_df.head(12)


SNR=0 dB | trial=1:   0%|          | 0/7247 [00:00<?, ?it/s][W921 19:35:22.058234706 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 92274688 bytes (free: 81002496, total: 4007591936).
[W921 19:35:23.043572069 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 125829120 bytes (free: 47448064, total: 4007591936).
SNR=0 dB | trial=1:   0%|          | 0/7247 [00:01<?, ?it/s, samples=20]
SNR=4 dB | trial=1:   0%|          | 0/7247 [00:01<?, ?it/s, samples=20]
SNR=8 dB | trial=1:   0%|          | 0/7247 [00:00<?, ?it/s][W921 19:35:26.556554799 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 148897792 bytes (free: 81002496, total: 4007591936).
SNR=8 dB | trial=1:   0%|          | 0/7247 [00:01<?, ?it/s, samples=20]
SNR=12 dB | trial=1:   0%|          | 0/7247 [00:01<?, ?it/s, samples=20]
SNR=16 dB | trial=1:   0%|          | 0/7247 [00

,id,snr_db,trial,language,source_en,reference,prediction
0,312325,0.0,1,en,to do so would be like saying to someone bent ...,to do so would be like saying to someone bent ...,"no, it would like to say to say to the united ..."
1,1354636,0.0,1,en,it is my belief that we need a special represe...,it is my belief that we need a special represe...,"it is a use of an, an bad bad of that the very..."
2,1004555,0.0,1,en,with regard to the research framework programm...,with regard to the research framework programm...,"in favour of the new funding, we are concerned..."
3,1388064,0.0,1,en,emu also rules out an economy that is sustaina...,emu also rules out an economy that is sustaina...,this is also a basis of the european union is ...
4,889116,0.0,1,en,"this fund, as you, commissioner, and my fellow...","this fund, as you, commissioner, and my fellow...","mr president, as you are the commission, i hav..."
5,996197,0.0,1,en,this is why the 'women in science' research pr...,this is why the 'women in science' research pr...,"this is the commission, in former the converge..."
6,259151,0.0,1,en,the issue of a secure long-term energy supply ...,the issue of a secure long-term energy supply ...,the report is the very long-term european unio...
7,531406,0.0,1,en,rural villages quickly become dormitory towns ...,rural villages quickly become dormitory towns ...,"rural states to be further, the authorities ha..."
8,1305365,0.0,1,en,item 12 about the community system on fisherie...,item 12 about the community system on fisherie...,the year of the european year for the eu and f...
9,1110804,0.0,1,en,"nevertheless, member states executed only 2 25...","nevertheless, member states executed only 2 25...","however, the member states have any 2 2 of oth..."


## 10. Metrics — repository `DeepSCMetrics`

No custom BLEU/semantic metric implementation is needed here.  
The branch `all` already provides corpus BLEU, chrF++, semantic similarity, ROUGE-L, exact match and BPE token accuracy.


In [12]:
if COMPUTE_METRICS:

    metrics = DeepSCMetrics(
        sp=sp,
        semantic_model=SEMANTIC_MODEL,
        device=str(device),
    )

    metric_rows = []

    group_columns = [
        "snr_db",
        "trial",
        "language",
    ]

    for keys, group in predictions_df.groupby(
        group_columns
    ):

        snr_db, trial, lang = keys

        result = metrics.evaluate_corpus(
            references=group["reference"].tolist(),
            hypotheses=group["prediction"].tolist(),
        )

        metric_rows.append(
            {
                "snr_db": snr_db,
                "trial": trial,
                "language": lang,
                **result.as_dict(),
                "num_samples": len(group),
            }
        )

    metrics_df = pd.DataFrame(
        metric_rows
    )

    display(metrics_df)

else:

    metrics_df = pd.DataFrame()

    print(
        "Metrics disabled. "
        "Set COMPUTE_METRICS = True to enable them."
    )


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13824.70it/s]
[W921 19:37:01.687510163 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 769654784 bytes (free: 479461376, total: 4007591936).


,snr_db,trial,language,bleu,chrf,semantic_similarity,rouge_l,exact_match_rate,token_accuracy,num_samples
0,0.0,1,en,4.849541,26.385830,0.418296,0.252225,0.00,0.087235,20
1,0.0,1,es,7.197608,28.120321,0.422976,0.245533,0.00,0.070274,20
2,0.0,1,fr,4.245661,25.593475,0.408960,0.212161,0.00,0.049968,20
3,0.0,1,pt,4.293282,24.990930,0.415605,0.235997,0.00,0.057624,20
4,4.0,1,en,63.080333,70.592957,0.768674,0.799901,0.05,0.691235,20
5,4.0,1,es,20.798968,46.504103,0.661020,0.485009,0.00,0.178745,20
6,4.0,1,fr,19.345836,43.913180,0.657738,0.441162,0.00,0.157265,20
7,4.0,1,pt,20.332611,47.182538,0.652319,0.491585,0.00,0.185884,20
8,8.0,1,en,66.517073,73.163776,0.767529,0.808860,0.15,0.670357,20
9,8.0,1,es,22.672144,47.492515,0.688579,0.505307,0.00,0.206071,20


## 11. Inspect the same sample in four languages


In [13]:
if not predictions_df.empty:

    example_id = predictions_df.iloc[0]["id"]

    display(
        predictions_df[
            predictions_df["id"] == example_id
        ][
            [
                "snr_db",
                "trial",
                "source_en",
                "language",
                "reference",
                "prediction",
            ]
        ]
    )


,snr_db,trial,source_en,language,reference,prediction
0,0.0,1,to do so would be like saying to someone bent ...,en,to do so would be like saying to someone bent ...,"no, it would like to say to say to the united ..."
20,0.0,1,to do so would be like saying to someone bent ...,pt,fazer isso seria como dizer a alguém decidido ...,"não, gostaria de dizer que, em vez, a união eu..."
40,0.0,1,to do so would be like saying to someone bent ...,es,hacerlo sería como decirle a alguien decidido ...,"no, no, no me gustaría decir que el estado de ..."
60,0.0,1,to do so would be like saying to someone bent ...,fr,ce serait comme dire à quelqu'un déterminé à t...,il ne s'agit pas de dire que nous devons dire ...
80,4.0,1,to do so would be like saying to someone bent ...,en,to do so would be like saying to someone bent ...,"to do so would be like to say to benten back, ..."
100,4.0,1,to do so would be like saying to someone bent ...,pt,fazer isso seria como dizer a alguém decidido ...,"para fazer isso seria, por dizer, em vez de di..."
120,4.0,1,to do so would be like saying to someone bent ...,es,hacerlo sería como decirle a alguien decidido ...,"por lo tanto, me gustaría decir que el ben de ..."
140,4.0,1,to do so would be like saying to someone bent ...,fr,ce serait comme dire à quelqu'un déterminé à t...,"pour faire, en tant que président, je tiens à ..."
160,8.0,1,to do so would be like saying to someone bent ...,en,to do so would be like saying to someone bent ...,to do so would be like to say to benting at sp...
180,8.0,1,to do so would be like saying to someone bent ...,pt,fazer isso seria como dizer a alguém decidido ...,"para fazer isso seria dizer que, em vez, ""aar-..."


## 12. Save outputs

In [ ]:
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)

predictions_path = os.path.join(
    OUTPUT_DIR,
    "predictions.csv",
)

metrics_path = os.path.join(
    OUTPUT_DIR,
    "metrics.csv",
)

predictions_df.to_csv(
    predictions_path,
    index=False,
)

if not metrics_df.empty:
    metrics_df.to_csv(
        metrics_path,
        index=False,
    )

print("Saved:", predictions_path)

if not metrics_df.empty:
    print("Saved:", metrics_path)


## Why this notebook is preferable

This notebook does not implement a second version of the DeepSC data, BPE, channel, decoder, checkpoint, or metric logic.

If those functions change in branch `all`, the notebook automatically uses the repository version after the next kernel restart.

The experiment-specific guarantee remains:

> **one `send_through_channel()` call → same `Rx_sig` reused for all language-control tokens.**
